In [ ]:
!pip install ultralytics opencv-python tqdm --quiet

In [ ]:
from pathlib import Path
MODEL_PATH = '../models/yolov26n/weights/best.pt'
VIDEO_INPUT = '../static/video/rescuer.mp4'
VIDEO_OUTPUT = 'detected_output.mp4'
CONFIDENCE = 0.75
if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}. Train the model or adjust MODEL_PATH.')
if not Path(VIDEO_INPUT).exists():
    raise FileNotFoundError(f'Video not found: {VIDEO_INPUT}. Add the video to the project or adjust VIDEO_INPUT.')
print('Configuration OK.')

In [ ]:
import cv2
from ultralytics import YOLO
try:
    from tqdm import tqdm
    use_tqdm = True
except ImportError:
    use_tqdm = False
model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_INPUT)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps, (width, height))
pbar = tqdm(total=total_frames, desc='Processing', unit='frame') if use_tqdm else None
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, conf=CONFIDENCE, verbose=False)
    result_img = results[0].plot()
    out.write(result_img)
    frame_idx += 1
    if pbar:
        pbar.update(1)
    elif frame_idx % 30 == 0:
        print(f'Frames processed: {frame_idx}/{total_frames}')
if pbar:
    pbar.close()
cap.release()
out.release()
print(f'Video saved: {VIDEO_OUTPUT}')

In [ ]:
from IPython.display import Video
Video(VIDEO_OUTPUT, embed=True, width=640)